In [0]:
from datetime import datetime

 

from pyspark.sql import functions as F

from pyspark.sql.types import StructType, StructField, StringType

 

LANDING_PATH = "/Volumes/automotive_catalog/automotive_mfg/landing/quality_inspection"

ARCHIVE_PATH = "/Volumes/automotive_catalog/automotive_mfg/archive/quality_inspection"

BRONZE_TABLE = "automotive_catalog.automotive_mfg.bronze_quality_inspection"

 

def audit_log(message):

    print(f"[AUDIT] {message}")

 

schema = StructType([

    StructField("inspection_id", StringType(), True),

    StructField("vin_number", StringType(), True),

    StructField("inspection_station", StringType(), True),

    StructField("defect_category", StringType(), True),

    StructField("defect_severity", StringType(), True),

    StructField("rework_required", StringType(), True),

    StructField("inspection_date", StringType(), True),

    StructField("op_code", StringType(), True)

])

In [0]:
batch_run_id = f"batch_{datetime.utcnow().strftime('%Y%m%d%H%M%S')}"

# Landing / archive folders (no-op if they already exist)

dbutils.fs.mkdirs(LANDING_PATH)

dbutils.fs.mkdirs(ARCHIVE_PATH)

 

landing_files = [

    f.path for f in dbutils.fs.ls(LANDING_PATH)

    if f.name.lower().endswith(".csv")

]

if not landing_files:

    audit_log("No new quality_inspection files in landing - nothing to ingest")

else:

    audit_log(f"Batch {batch_run_id}: found {len(landing_files)} file(s) in landing")

 

    df = (

        spark.read

        .option("header", True)

        .schema(schema)

        .csv(LANDING_PATH)

        .withColumn("inspection_date", F.to_date(F.col("inspection_date")))

        .withColumn("ingest_ts", F.current_timestamp())

        .withColumn("source_file_name", F.col("_metadata.file_path"))

        .withColumn("batch_run_id", F.lit(batch_run_id))

        .withColumn("load_date", F.current_date())

    )

In [0]:
row_count = df.count()

df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)

audit_log(f"Loaded {row_count} rows into {BRONZE_TABLE}")

 

# Archive only after a successful write - this is what makes

# the file ingestion exactly-once per file.

for file_path in landing_files:

    file_name = file_path.rstrip("/").split("/")[-1]

    dbutils.fs.mv(file_path, f"{ARCHIVE_PATH}/{file_name}")

 

audit_log(f"Moved {len(landing_files)} file(s) to {ARCHIVE_PATH}")

In [0]:
%sql

select * from automotive_catalog.automotive_mfg.bronze_quality_inspection